In [1]:
#TODO make sure this renders in the github repo

# Llama 3 Tokenizer

🌟 A tokenizer takes raw text strings and outputs a 1D list of discrete integers/tokens (e.g., "The brown rabbit..." becomes [1, 45, 102, ...]).


**Useful Links:**

- [Tokenization Visualization](https://tiktokenizer.vercel.app/?model=meta-llama%2FMeta-Llama-3-8B) 
- [HuggingFace Byte-Pair Encoding tokenization explanation](https://huggingface.co/learn/llm-course/chapter6/5)

**Llama 3 paper:** "We use a vocabulary with $\bold{128K}$ tokens. Our token vocabulary combines $\bold{100K}$ tokens from the **tiktoken tokenizer** with $\bold{28K}$ additional tokens to better support non-English languages. Compared to the Llama $2$ tokenizer, our new tokenizer improves compression rates on a sample of English data from $3.17$ to $3.94$ characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding $\bold{28K}$ tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

- They took [OpenAI's pre-trained TikToken tokenizer](https://github.com/openai/tiktoken), then they got their own dataset consisting of non-English languages and code, ran a BPE training algorithm on that dataset to generate an additional 28K BPE merge rules, and finally combined the pre-trained tokenizer with the 28K tokens to get the Llama 3 tokenizer.
- Unfortunately for my scaled-down model, I can not use OpenAI's TikToken, instead I trained a custom BPE tokenizer.

**Goals:**

- [x] Use HuggingFace's Tokenizer library to create a tokenizer that will be trained on a the FineWeb-edu dataset.

In [2]:
import easyjupyter

In [3]:
# @i-c
%load_ext autoreload
%autoreload 2

In [4]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing
from transformers import PreTrainedTokenizerFast
import json

In [5]:
from typing import TYPE_CHECKING
if TYPE_CHECKING: 
    from configs import ConfigTemplate

In [6]:
class BPETokenizer:
    def __init__(self, cfg: ConfigTemplate):
        """Handles training, saving and loading a custom BPE tokenizer for my scaled down Llama model. Not to be used with the trained Llama 3 models, they have their own tokenizers."""
        self.cfg = cfg
        self.tokenizer_dir = cfg.artifacts_dir / "universal_tokenizer"

    def _get_path(self):
        """Returns the paths for the tokenizer file and info file."""
        samples = self.cfg.text_only.pretrain.initial_stage.tokenizer_training_samples
        base_name = f"bpe_vocab_size_{self.cfg.vocab_size}_samples_{samples}"
        return (
            self.tokenizer_dir / f"{base_name}.json",
            self.tokenizer_dir / f"{base_name}_info.json",
        )

    def get_hf_tokenizer(self) -> PreTrainedTokenizerFast:
        """Tell the tokenizer how it should format conversation when formatting the datasets for post-training."""
        path, _ = self._get_path()
        special_tokens = self.cfg.special_tokens

        bos_token = special_tokens["doc_begin_token"]["token"]
        eos_token = special_tokens["doc_end_token"]["token"]

        # Wrap the BPE JSON file in HuggingFace's wrapper
        tokenizer = PreTrainedTokenizerFast(
            tokenizer_file=str(path),
            bos_token=bos_token,
            eos_token=eos_token,
            pad_token=special_tokens["pad_token"]["token"],
            unk_token=special_tokens["unk_token"]["token"],
        )

        # Attach Jinja2 chat template
        tokenizer.chat_template = (
            "{% for message in messages %}"
            f"{{{{ '{bos_token}' + message['role'] + '\\n' + message['content'] + '{eos_token}' }}}}"
            "{% endfor %}"
        )
        return tokenizer

    def train(self, samples: list[str]) -> Tokenizer:
        """
        Train a custom BPE tokenizer.
        """
        chunk_size = 2000  # The number of documents that is sent to the Rust backend per iteration.
        special_tokens = self.cfg.special_tokens
        num_samples = (
            self.cfg.text_only.pretrain.initial_stage.tokenizer_training_samples
        )

        def batch_iterator():
            batch = []

            for text in samples:
                batch.append(text)
                if len(batch) == chunk_size:
                    yield batch
                    batch = []
            if batch:
                # Yield any remaining items
                yield batch

        self.tokenizer_dir.mkdir(parents=True, exist_ok=True)
        tokenizer_path, info_path = self._get_path()

        # Initialize a new Byte-Pair-Encoding tokenizer model.
        tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]["token"]))
        # Using Byte-Level pre-tokenization ensures that spaces and punctuation are handled correctly without losing characters.
        tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
        # Configure the BPE trainer
        trainer = BpeTrainer(
            vocab_size=self.cfg.vocab_size,
            special_tokens=[v["token"] for v in special_tokens.values()],
            initial_alphabet=ByteLevel.alphabet(),
        )

        # === Train the tokenizer on the raw text file
        print(
            f"Training the BPE tokenizer | Max Document Count: {num_samples} | chunk_size: {chunk_size}...\n"
        )
        tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

        # === Set up the post-processor to automatically add the begin_of_text token to sequences
        tokenizer.post_processor = TemplateProcessing(
            single=f"{special_tokens['doc_begin_token']['token']} $A",
            pair=f"{special_tokens['doc_begin_token']['token']} $A $B",
            special_tokens=[
                (
                    special_tokens["doc_begin_token"]["token"],
                    tokenizer.token_to_id(special_tokens["doc_begin_token"]["token"]),
                )
            ],
        )

        # Tell the tokenizer how to decode the token ids back into text, example: 'Ġ' back to ' '
        tokenizer.decoder = ByteLevelDecoder()

        # === Save the trained tokenizer to a JSON file
        tokenizer.save(str(tokenizer_path))
        print(f"Tokenizer saved to {tokenizer_path}")

        with open(info_path, "w") as f:
            config_dict = {
                "name": tokenizer_path.name,
                "description": "Custom Llama BPE Tokenizer",
                "vocab_size": self.cfg.vocab_size,
                "num_samples": num_samples,
                "chunk_size": chunk_size,
            }
            json.dump(config_dict, f, indent=4)

        return tokenizer

    def load_tokenizer(self) -> tuple[bool, Tokenizer]:
        """Load a tokenizer based on the vocab size and the maximum number of documents (num_samples) it was trained on."""
        num_samples = (
            self.cfg.text_only.pretrain.initial_stage.tokenizer_training_samples
        )
        path, _ = self._get_path()

        if path.exists():
            print(
                f"\nLoading existing trained BPE Tokenizer with vocab size: {self.cfg.vocab_size} "
                f"(Trained on {num_samples} documents) from: ({path})..."
            )
            tokenizer = Tokenizer.from_file(str(path))
            return True, tokenizer
        return False, None

### Test: BPETokenizer()

In [7]:
# @i-c
from configs import Llama3_1_405B  # TODO change to my scaled down model
from datasets import load_dataset

cfg = Llama3_1_405B.initialize(is_overfit=True)

ds_stream = load_dataset(
    cfg.text_only.pretrain.initial_stage.dataset.name,
    cfg.text_only.pretrain.initial_stage.dataset.config,
    split="train",
    streaming=True,
)
samples = [
    sample["text"]
    for sample in ds_stream.take(
        cfg.text_only.pretrain.initial_stage.tokenizer_training_samples
    )
]
print("\n\nTesting training a tokenizer...")
t = BPETokenizer(cfg)
tokenizer = t.train(samples)


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: Llama 3.1 405B
Configured for overfitting...


Testing training a tokenizer...
Training the BPE tokenizer | Max Document Count: 100 | chunk_size: 2000...




Tokenizer saved to /Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_128000_samples_100.json


In [8]:
# @i-c
# Test: Loading A Trained Tokenizer
success, loaded_tokenizer = t.load_tokenizer()
if success:
    print("Tokenizer successfully trained!")

    print("\n\nEncoding a string to IDs")
    text_str = "The brown rabbit ate the apple."
    encoded = loaded_tokenizer.encode(text_str)
    print(f"Original string: {text_str}")
    print(f"Token IDs: {encoded.ids}")
    print(f"Tokens: {encoded.tokens}")

    print("\n\nDecoding IDs to a string")
    decoded = loaded_tokenizer.decode(encoded.ids)
    print(f"Decoded string: {decoded}")


Loading existing trained BPE Tokenizer with vocab size: 128000 (Trained on 100 documents) from: (/Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_128000_samples_100.json)...
Tokenizer successfully trained!


Encoding a string to IDs
Original string: The brown rabbit ate the apple.
Token IDs: [1, 456, 6906, 743, 69, 4358, 8321, 266, 13572, 17]
Tokens: ['<|begin_of_doc|>', 'The', 'Ġbrown', 'Ġra', 'b', 'bit', 'Ġate', 'Ġthe', 'Ġapple', '.']


Decoding IDs to a string
Decoded string: The brown rabbit ate the apple.
